In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.metrics import r2_score
import numpy as np

In [ ]:
X,y=load_diabetes(return_X_y=True)

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=4)

In [ ]:
from sklearn.linear_model import Ridge

In [ ]:
reg=Ridge(alpha=0.1,solver='cholesky')
#solver : which method to use to solve ridge regression

solver{‘auto’, ‘svd’, ‘cholesky’, ‘lsqr’, ‘sparse_cg’, ‘sag’, ‘saga’, ‘lbfgs’}, default=’auto’
Solver to use in the computational routines:

‘auto’ chooses the solver automatically based on the type of data.

‘svd’ uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than ‘cholesky’ at the cost of being slower.

‘cholesky’ uses the standard scipy.linalg.solve function to obtain a closed-form solution.

‘sparse_cg’ uses the conjugate gradient solver as found in scipy.sparse.linalg.cg. As an iterative algorithm, this solver is more appropriate than ‘cholesky’ for large-scale data (possibility to set tol and max_iter).

‘lsqr’ uses the dedicated regularized least-squares routine scipy.sparse.linalg.lsqr. It is the fastest and uses an iterative procedure.

‘sag’ uses a Stochastic Average Gradient descent, and ‘saga’ uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that ‘sag’ and ‘saga’ fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from sklearn.preprocessing.

‘lbfgs’ uses L-BFGS-B algorithm implemented in scipy.optimize.minimize. It can be used only when positive is True.

In [ ]:
reg.fit(X_train,y_train)

Ridge(alpha=0.1, solver='cholesky')

In [ ]:
reg.fit(X_train,y_train)

Ridge(alpha=0.1, solver='cholesky')

In [ ]:
y_pred=reg.predict(X_test)
r2_score(y_test,y_pred)

0.4693128853309805

In [ ]:
print(reg.coef_)
print(reg.intercept_)

[  44.02063391 -241.69329987  452.98665299  332.04420177  -76.33686744
  -68.52143809 -164.98809083  149.96908118  431.61347417   58.51862681]
150.89053425602808


In [ ]:
#In sklearn they use Identity matrix but keep the first i.e. I[0][0]=0
class MeraRidge:

  def __init__(self,alpha=0.1):
    self.alpha=alpha
    self.coef_=None
    self.intercept_=None

  def fit(self,X_train,y_train):
    X_train=np.insert(X_train,0,1,axis=1)
    I=np.identity(X_train.shape[1])
    result=np.linalg.inv(np.dot(X_train.T,X_train)+self.alpha*I).dot(X_train.T).dot(y_train)

    self.intercept_=result[0]
    self.coef_=result[1:]

    print(self.intercept_)
    print(self.coef_)

  def predict(self,X_test):
    return np.dot(X_test,self.coef_) + self.intercept_






In [ ]:
reg=MeraRidge()

In [ ]:
reg.fit(X_train,y_train)

150.84762720023167
[  44.04252774 -241.72019592  452.99153345  332.06044266  -76.32617866
  -68.54884462 -165.01664054  149.93980143  431.59483654   58.54419942]


In [ ]:
y_pred=reg.predict(X_test)
r2_score(y_test,y_pred)

0.46921361559931885

## Ridge Regression using Gradient Descent

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=4)

### Using SGDRegressor

In [ ]:
from sklearn.linear_model import SGDRegressor

In [ ]:
reg=SGDRegressor(penalty='l2',max_iter=500,eta0=0.1,learning_rate='constant',alpha=0.01)

In [ ]:
reg.fit(X_train,y_train)

SGDRegressor(alpha=0.01, eta0=0.1, learning_rate='constant', max_iter=500)

In [ ]:
y_pred=reg.predict(X_test)
print("R2Score: ",r2_score(y_test,y_pred))
print(reg.coef_)
print(reg.intercept_)

R2Score:  0.24441578175651058
[ 36.93570048 -19.21358788 135.3772727  105.73056066  26.37914037
  11.12651659 -82.49331974  82.08963389 130.59723029  65.16464277]
[169.45643807]


### Using Ridge

In [ ]:
from sklearn.linear_model import Ridge
reg=Ridge(alpha=0.001,max_iter=500,solver='sparse_cg')

In [ ]:
reg.fit(X_train,y_train)

Ridge(alpha=0.001, max_iter=500, solver='sparse_cg')

In [ ]:
y_pred=reg.predict(X_test)
print("R2Score: ",r2_score(y_test,y_pred))
print(reg.coef_)
print(reg.intercept_)

R2Score:  0.46250101619914563
[  34.52192544 -290.84084076  482.40181344  368.0678662  -852.44873179
  501.59160336  180.11115788  270.76333979  759.73534372   37.4913546 ]
151.10198517439466


### Using Class

In [ ]:
class MeraRidgeGD:

    def __init__(self,epochs,learning_rate,alpha):

        self.learning_rate = learning_rate
        self.epochs = epochs
        self.alpha = alpha
        self.coef_ = None
        self.intercept_ = None

    def fit(self,X_train,y_train):

        self.coef_ = np.ones(X_train.shape[1])
        self.intercept_ = 0
        thetha = np.insert(self.coef_,0,self.intercept_)

        X_train = np.insert(X_train,0,1,axis=1)

        for i in range(self.epochs):
            thetha_der = 2*(np.dot(X_train.T,X_train).dot(thetha) - np.dot(X_train.T,y_train) + self.alpha*thetha)/X_train.shape[0]
            thetha = thetha - self.learning_rate*thetha_der

        self.coef_ = thetha[1:]
        self.intercept_ = thetha[0]

    def predict(self,X_test):

        return np.dot(X_test,self.coef_) + self.intercept_


In [ ]:
reg=MeraRidgeGD(epochs=500,alpha=0.001,learning_rate=0.5)
reg.fit(X_train,y_train)

y_pred=reg.predict(X_test)
print("R2Score: ",r2_score(y_test,y_pred))
print(reg.coef_)
print(reg.intercept_)

R2Score:  0.4612784697393768
[  56.9733675  -146.60755807  387.26828892  283.9313164     2.20183378
  -54.4164052  -183.1824496   152.31528053  345.25123827  120.99629533]
151.06619219020112
